# Full Fine-Tuning Evaluation

Evaluation of eight training conditions across seeds 42–46.

- **CBIS-DDSM:** internal test split
- **BCDR:** external test dataset
- Checkpoint selection based on the CBIS-DDSM validation loss


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", None)

ROOT = Path("../../extension/hybrid_experiments/full_ft").resolve()
EXPORT_DIR = ROOT / "evaluation_exports"
EXPORT_DIR.mkdir(exist_ok=True)

EXPERIMENTS = [
    "real_only",
    "hybrid_025",
    "hybrid_050",
    "hybrid_100",
    "hybrid_200",
    "hybrid_250",
"synthetic_only_matched",
"synthetic_only_maximum",
]
SEEDS = [42, 43, 44, 45, 46]
DATASETS = ["CBIS-DDSM", "BCDR"]
METRICS = [
    "loss",
    "accuracy",
    "Precision",
    "Recall",
    "F1-Score",
    "AUROC",
    "AUPRC",
]

print("Results directory:", ROOT)
print("Export directory:", EXPORT_DIR)


In [ ]:
records = []

for experiment in EXPERIMENTS:
    for seed in SEEDS:
        run_dir = ROOT / f"{experiment}_seed{seed}"
        logs = list(run_dir.glob("*.txt"))

        if len(logs) != 1:
            raise RuntimeError(
                f"{experiment}, Seed {seed}: "
                f"found {len(logs)} logs instead of exactly one"
            )

        text = logs[0].read_text(errors="replace")

        best_match = re.search(
            r"Best model was achieved after (\d+) epochs, "
            r"with val loss = ([0-9.eE+-]+)",
            text,
        )
        if not best_match:
            raise RuntimeError(
                f"{experiment}, seed {seed}: best-checkpoint record missing"
            )

        epoch = int(best_match.group(1))
        val_loss = float(best_match.group(2))

        values_by_metric = {}
        for metric in METRICS:
            values = re.findall(
                rf"test {re.escape(metric)} in {epoch} epoch:"
                rf"\s+([0-9.eE+-]+)",
                text,
            )

            if len(values) != 2:
                raise RuntimeError(
                    f"{experiment}, Seed {seed}, {metric}: "
                    f"found {len(values)} values instead of two test values"
                )

            values_by_metric[metric] = list(map(float, values))

        for dataset_index, dataset in enumerate(DATASETS):
            record = {
                "experiment": experiment,
                "seed": seed,
                "dataset": dataset,
                "best_epoch": epoch,
                "val_loss": val_loss,
                "logfile": str(logs[0].relative_to(ROOT)),
            }

            for metric in METRICS:
                record[metric] = values_by_metric[metric][dataset_index]

            records.append(record)

results = pd.DataFrame(records)
results["experiment"] = pd.Categorical(
    results["experiment"],
    categories=EXPERIMENTS,
    ordered=True,
)
results["dataset"] = pd.Categorical(
    results["dataset"],
    categories=DATASETS,
    ordered=True,
)
results = results.sort_values(
    ["dataset", "experiment", "seed"]
).reset_index(drop=True)

assert len(results) == 80
assert results.groupby(["dataset", "experiment"], observed=True).size().eq(5).all()

results.to_csv(EXPORT_DIR / "full_ft_results_per_seed.csv", index=False)

print(f"{len(results)} result rows successfully loaded.")
print("This corresponds to 40 runs × 2 test datasets.")
results.head(10)


In [ ]:
check = (
    results.groupby(
        ["dataset", "experiment"],
        observed=True,
    )["seed"]
    .agg(["count", "min", "max"])
)
check


## Aggregate Results

The following tables report the mean and sample standard deviation across five seeds for each training condition.


In [ ]:
SUMMARY_METRICS = [
    "accuracy",
    "Precision",
    "Recall",
    "F1-Score",
    "AUROC",
    "AUPRC",
]

summary = (
    results.groupby(
        ["dataset", "experiment"],
        observed=True,
    )[SUMMARY_METRICS]
    .agg(["mean", "std"])
)

summary.to_csv(EXPORT_DIR / "full_ft_summary.csv")
summary.round(4)


In [ ]:
formatted_rows = []

for (dataset, experiment), group in results.groupby(
    ["dataset", "experiment"],
    observed=True,
):
    row = {
        "Dataset": dataset,
        "Training condition": experiment,
    }

    for metric in SUMMARY_METRICS:
        row[metric] = (
            f"{group[metric].mean():.4f} ± "
            f"{group[metric].std(ddof=1):.4f}"
        )

    formatted_rows.append(row)

formatted_summary = pd.DataFrame(formatted_rows)
formatted_summary


## Performance Across Training Conditions

Dots represent individual seeds. Large markers and error bars show the mean and sample standard deviation.


In [ ]:
DISPLAY_NAMES = {
    "real_only": "Real only",
    "hybrid_025": "Hybrid 25%",
    "hybrid_050": "Hybrid 50%",
    "hybrid_100": "Hybrid 100%",
    "hybrid_200": "Hybrid 200%",
    "hybrid_250": "Hybrid 250%",
"synthetic_only_matched": "Synthetic only (matched)",
"synthetic_only_maximum": "Synthetic only (maximum)",
}

PLOT_METRICS = ["accuracy", "F1-Score", "AUROC", "AUPRC"]

plot_data = results.copy()
plot_data["Training condition"] = (
    plot_data["experiment"].astype(str).map(DISPLAY_NAMES)
)

fig, axes = plt.subplots(
    2,
    2,
    figsize=(18, 10),
    sharex=True,
)

for ax, metric in zip(axes.flat, PLOT_METRICS):
    sns.stripplot(
        data=plot_data,
        x="Training condition",
        y=metric,
        hue="dataset",
        dodge=True,
        alpha=0.65,
        size=6,
        ax=ax,
    )

    sns.pointplot(
        data=plot_data,
        x="Training condition",
        y=metric,
        hue="dataset",
        dodge=0.35,
        errorbar="sd",
        join=False,
        markers="D",
        scale=0.9,
        ax=ax,
    )

    ax.set_title(metric)
    ax.set_xlabel("")
    ax.set_ylabel(metric)
    ax.tick_params(axis="x", rotation=35)

    if ax is not axes.flat[0]:
        ax.get_legend().remove()

handles, labels = axes.flat[0].get_legend_handles_labels()
axes.flat[0].get_legend().remove()

fig.legend(
    handles[:2],
    labels[:2],
    title="Test dataset",
    loc="upper center",
    ncol=2,
)

fig.suptitle(
    "Full Fine-Tuning Performance Across Eight Training Conditions",
    fontsize=16,
    y=1.02,
)
fig.tight_layout()

figure_path = EXPORT_DIR / "full_ft_main_metrics.png"
fig.savefig(figure_path, dpi=300, bbox_inches="tight")
plt.show()

print("Figure saved to:", figure_path)


## Paired Differences Relative to Real-Only Training

Each hybrid run is compared with the real-only run using the same random seed. Positive values indicate an improvement over the real-only baseline.


In [ ]:
baseline = (
    results[results["experiment"] == "real_only"]
    .set_index(["dataset", "seed"])
)

paired_records = []

for experiment in EXPERIMENTS[1:]:
    hybrid = (
        results[results["experiment"] == experiment]
        .set_index(["dataset", "seed"])
    )

    for dataset in DATASETS:
        for seed in SEEDS:
            for metric in SUMMARY_METRICS:
                paired_records.append({
                    "dataset": dataset,
                    "experiment": experiment,
                    "seed": seed,
                    "metric": metric,
                    "difference": (
                        hybrid.loc[(dataset, seed), metric]
                        - baseline.loc[(dataset, seed), metric]
                    ),
                })

paired = pd.DataFrame(paired_records)
paired.to_csv(
    EXPORT_DIR / "full_ft_paired_differences.csv",
    index=False,
)

paired.head()


In [ ]:
selected = paired[
    paired["metric"].isin(["AUROC", "AUPRC"])
].copy()

selected["Training condition"] = (
    selected["experiment"].map(DISPLAY_NAMES)
)

g = sns.catplot(
    data=selected,
    x="Training condition",
    y="difference",
    hue="dataset",
    col="metric",
    kind="point",
    errorbar="sd",
    dodge=0.35,
    markers="D",
    height=5,
    aspect=1.15,
)

for ax in g.axes.flat:
    ax.axhline(0, color="black", linestyle="--", linewidth=1)
    ax.tick_params(axis="x", rotation=35)
    ax.set_xlabel("")
    ax.set_ylabel("Paired difference vs. real only")

g.set_titles("{col_name}")
g.figure.suptitle(
    "Paired Performance Differences Relative to Real-Only Training",
    y=1.05,
)

figure_path = EXPORT_DIR / "full_ft_paired_differences.png"
g.figure.savefig(figure_path, dpi=300, bbox_inches="tight")
plt.show()

print("Figure saved to:", figure_path)


## Best Checkpoint Epochs

The checkpoint epoch and validation loss are properties of each training run and are therefore shown once per run.


In [ ]:
run_information = (
    results[
        results["dataset"] == "CBIS-DDSM"
    ][
        [
            "experiment",
            "seed",
            "best_epoch",
            "val_loss",
            "logfile",
        ]
    ]
    .sort_values(["experiment", "seed"])
    .reset_index(drop=True)
)

run_information.to_csv(
    EXPORT_DIR / "full_ft_run_information.csv",
    index=False,
)

run_information


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epoch_plot = run_information.copy()
epoch_plot["Training condition"] = (
    epoch_plot["experiment"].astype(str).map(DISPLAY_NAMES)
)

sns.stripplot(
    data=epoch_plot,
    x="Training condition",
    y="best_epoch",
    color="tab:blue",
    size=7,
    ax=axes[0],
)
sns.pointplot(
    data=epoch_plot,
    x="Training condition",
    y="best_epoch",
    color="black",
    errorbar="sd",
    join=False,
    markers="D",
    ax=axes[0],
)
axes[0].set_title("Selected Checkpoint Epoch")
axes[0].set_xlabel("")
axes[0].set_ylabel("Best epoch")

sns.stripplot(
    data=epoch_plot,
    x="Training condition",
    y="val_loss",
    color="tab:orange",
    size=7,
    ax=axes[1],
)
sns.pointplot(
    data=epoch_plot,
    x="Training condition",
    y="val_loss",
    color="black",
    errorbar="sd",
    join=False,
    markers="D",
    ax=axes[1],
)
axes[1].set_title("Validation Loss at Selected Checkpoint")
axes[1].set_xlabel("")
axes[1].set_ylabel("Validation loss")

for ax in axes:
    ax.tick_params(axis="x", rotation=35)

fig.tight_layout()

figure_path = EXPORT_DIR / "full_ft_checkpoint_selection.png"
fig.savefig(figure_path, dpi=300, bbox_inches="tight")
plt.show()

print("Figure saved to:", figure_path)
